# 🧊 SAR Sea Ice — Baseline Comparison (for Information Fusion)

Compares your model against published language-/reasoning-segmentation models,
**all evaluated on the same 90 SAR test images**, with identical metric code
(gIoU / cIoU / Dice). Mirrors Table 1 of the underwater reasoning-seg paper.

### ⚠️ How to use this notebook
These models need **incompatible `transformers` versions**, so they **cannot**
all run in one session. The workflow is:

1. **Run Section 0** (shared setup) — always, after every restart.
2. Run **one model section** at a time. Each saves its result to a JSON on Drive.
3. For the heavy models (LISA, GeoPixel, SEEM): **restart the runtime**
   (`Runtime → Restart session`), re-run Section 0, then run that model's section.
4. When all desired sections are done, run **Section 8 (Aggregate)** to build
   the comparison table + LaTeX from whatever JSONs exist on Drive.

Every number is from real inference on your data — nothing is copied from other
papers (those used different datasets, so copying would be invalid).

| Section | Model | Type | transformers | Session |
|---|---|---|---|---|
| 1 | **Your model** | SAR-trained, text-guided | new | shared |
| 2 | CLIPSeg | zero-shot text→mask | new | shared |
| 3 | lang-sam (GroundingDINO+SAM) | zero-shot | new | shared |
| 4 | DeepLabv3+ | trained, no text | new | shared |
| 5 | LISA-7B | zero-shot reasoning | **old (~4.31)** | **own** |
| 6 | GeoPixel | zero-shot RS-LMM | pinned | **own** |
| 7 | SEEM | zero-shot | pinned | **own** |
| 8 | **Aggregate → table + LaTeX** | — | any | shared |


## 0. 🔧 Shared Setup — RUN THIS FIRST (after every restart)

In [ ]:
# Mount Drive, clone repo, define the test-set lister + metric harness.
import os, subprocess, sys, json
import numpy as np
from pathlib import Path
from PIL import Image

# ── Drive ────────────────────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/sea_ice_seg'
except Exception as e:
    print('Drive mount skipped:', e)
    DRIVE_DIR = '/content/sea_ice_seg_drive'

COMPARE_DIR = f'{DRIVE_DIR}/outputs/comparison'
os.makedirs(COMPARE_DIR, exist_ok=True)
print('Comparison results dir:', COMPARE_DIR)

# ── Repo ─────────────────────────────────────────────────────────────────────
REPO_URL = 'https://github.com/prakhar443/sea_ice_seg.git'
BRANCH   = 'claude/laughing-thompson-AhCX7'
REPO_DIR = '/content/sea_ice_seg'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# ── Install repo dependencies + fix Colab's torchao/peft conflict ─────────────
# Colab ships torchao 0.10.0, but peft demands torchao>=0.16.0 (which needs a
# newer torch than Colab has, so it won't upgrade). The pipeline never uses
# torchao — peft only probes for it. UNINSTALLING torchao makes peft skip the
# version check gracefully (is_torchao_available() returns False), avoiding the
# ImportError. This is more reliable than trying to upgrade torchao.
subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'], check=False)
subprocess.run([sys.executable,'-m','pip','install','-q',
    'transformers>=4.40.0','peft>=0.10.0','accelerate>=0.27.0',
    'einops>=0.7.0','timm>=0.9.0','scikit-learn','pandas'], check=False)

# Belt-and-suspenders: if torchao is still importable in THIS kernel from a
# prior import, hide it so peft's find_spec check returns None.
import importlib.util
if importlib.util.find_spec('torchao') is not None:
    print('⚠️  torchao still present — if peft still errors, RESTART the runtime '
          'and run this cell again before Section 1.')

DATA_ROOT = f'{REPO_DIR}/dataset'

# ── Deterministic test-set lister (matches the training split exactly) ────────
def list_test_set(data_root=DATA_ROOT, seed=42, train_ratio=0.70, val_ratio=0.15):
    from config import ICE_CLASSES, ICE_CLASS_TO_IDX
    data_root = Path(data_root); samples = []
    for cls in ICE_CLASSES:
        imgs_dir = data_root / cls / 'images'
        mask_dir = data_root / cls / 'masks'
        if not imgs_dir.exists():
            continue
        files = sorted(imgs_dir.glob('*.jpg'))
        rng = np.random.default_rng(seed + ICE_CLASS_TO_IDX[cls])
        order = rng.permutation(len(files)).tolist()
        n = len(files); ntr = round(n*train_ratio); nv = round(n*val_ratio)
        for i in order[ntr+nv:]:
            # Mask naming convention: '1_1003_.jpg' -> '1_1003_scat.jpg'
            stem = files[i].stem.rstrip('_') + '_scat'
            mp = mask_dir / (stem + files[i].suffix)
            if not mp.exists():
                mp = mask_dir / files[i].name      # fallback
            samples.append((files[i], mp, cls))
    return samples

# ── Metric harness — identical for every model ────────────────────────────────
# predict_fn(pil_rgb) must return a HxW boolean/0-1 numpy mask (any resolution).
# Query for all text models: "sea ice".
def evaluate_predictor(predict_fn, eval_size=512, verbose=True):
    samples = list_test_set()
    inter_tot = union_tot = 0.0
    ious, dices = [], []
    for k, (img_path, mask_path, cls) in enumerate(samples):
        pil = Image.open(img_path).convert('RGB')
        pred = np.asarray(predict_fn(pil))
        if pred.dtype != bool:
            pred = pred > 0.5
        pred = np.array(Image.fromarray((pred.astype('uint8')*255))
                        .resize((eval_size, eval_size), Image.NEAREST)) > 127
        if mask_path.exists():
            from data.dataset import binarize_mask
            g  = np.array(Image.open(mask_path).convert('L'))
            gb = binarize_mask(g, mode='otsu')          # per-image Otsu {0,1}
            gt = np.array(Image.fromarray((gb*255).astype('uint8'))
                          .resize((eval_size, eval_size), Image.NEAREST)) > 127
        else:
            gt = np.zeros((eval_size, eval_size), bool)
        inter = np.logical_and(pred, gt).sum()
        union = np.logical_or(pred, gt).sum()
        iou  = inter/union if union > 0 else (1.0 if pred.sum()==0 else 0.0)
        dice = 2*inter/(pred.sum()+gt.sum()) if (pred.sum()+gt.sum())>0 else 1.0
        ious.append(iou); dices.append(dice)
        inter_tot += inter; union_tot += union
        if verbose and (k+1) % 15 == 0:
            print(f'  {k+1}/{len(samples)}  running gIoU={np.mean(ious):.4f}')
    return {'gIoU': float(np.mean(ious)),
            'cIoU': float(inter_tot/union_tot) if union_tot > 0 else 0.0,
            'Dice': float(np.mean(dices)),
            'n': len(samples)}

def save_result(name, res):
    p = f'{COMPARE_DIR}/{name}.json'
    json.dump({'model': name, **res}, open(p, 'w'), indent=2)
    print(f'\n✅ {name}: gIoU={res["gIoU"]:.4f}  cIoU={res["cIoU"]:.4f}  '
          f'Dice={res["Dice"]:.4f}  (n={res["n"]})  → saved {p}')

print('\n✅ Section 0 ready. Test images:', len(list_test_set()))


## 1. 🏆 Your Model (SAR-trained, text-guided)

Runs your trained baseline pipeline (`best_model.pth`) through the **same**
harness so its gIoU/cIoU/Dice are directly comparable to every baseline.


In [ ]:
import torch, numpy as np
from PIL import Image

# Locate the trained checkpoint
CKPT = None
for c in [f'{DRIVE_DIR}/outputs/best_model.pth', f'{REPO_DIR}/outputs/best_model.pth']:
    if os.path.exists(c): CKPT = c; break
assert CKPT, 'best_model.pth not found in Drive/outputs or repo/outputs'
print('Checkpoint:', CKPT)

from config import cfg
from models.pipeline import SeaIceSegmentationPipeline
from data.preprocessing import SARPreprocessor

device = torch.device('cuda')

# Load checkpoint first and auto-detect the U-Net base width it was trained with.
# config.py defaults to decoder_base_channels=32, but the A100 run used 48.
# Reading mask_decoder.enc1.0.weight's out-channels gives the exact base.
ck = torch.load(CKPT, map_location='cpu')
sd = ck.get('model_state_dict', ck)
if 'mask_decoder.enc1.0.weight' in sd:
    base = sd['mask_decoder.enc1.0.weight'].shape[0]
    cfg.model.decoder_base_channels = int(base)
    print(f'Detected U-Net base width from checkpoint: {base}')

model = SeaIceSegmentationPipeline(cfg.model).to(device).eval()
missing, unexpected = model.load_state_dict(sd, strict=False)
print(f'Loaded checkpoint: {len(missing)} missing, {len(unexpected)} unexpected keys')

pre = SARPreprocessor(cfg.data)
GENERIC_TEXT = 'sea ice in synthetic aperture radar image'

@torch.no_grad()
def predict_ours(pil):
    # Match training EXACTLY: dataset feeds (gray/255.0) numpy to SARPreprocessor
    # (data/dataset.py line ~402), then forward(images, descriptions).
    arr = np.array(pil.convert('L')).astype('float32') / 255.0
    img = pre(arr).unsqueeze(0).to(device)
    out = model(img, [GENERIC_TEXT])
    m = out['masks'] if 'masks' in out else torch.sigmoid(out['mask_logits'])
    return m.squeeze().float().cpu().numpy() > 0.5

res = evaluate_predictor(predict_ours)
save_result('OursTextGuided', res)


## 2. CLIPSeg (zero-shot text→mask)

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers>=4.40.0'])

import torch, numpy as np
from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation
device = torch.device('cuda')
_proc = CLIPSegProcessor.from_pretrained('CIDAS/clipseg-rd64-refined')
_cs   = CLIPSegForImageSegmentation.from_pretrained('CIDAS/clipseg-rd64-refined').to(device).eval()

@torch.no_grad()
def predict_clipseg(pil):
    inp = _proc(text=['sea ice'], images=[pil], return_tensors='pt').to(device)
    logits = _cs(**inp).logits           # (352,352)
    prob = torch.sigmoid(logits).squeeze().cpu().numpy()
    return prob > 0.5

res = evaluate_predictor(predict_clipseg)
save_result('CLIPSeg', res)


## 3. lang-sam — GroundingDINO + SAM (zero-shot)

Represents the language-prompted Segment-Anything family (= Grounded-SAM).


In [ ]:
# GroundingDINO + SAM via transformers (robust; same method as lang-sam).
# The lang-sam PyPI package has a fragile dep tree on Colab, so we build the
# identical GroundingDINO->boxes->SAM->masks pipeline with pure transformers.
import subprocess, sys
subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers>=4.40.0'])

import torch, numpy as np
from transformers import (AutoProcessor, AutoModelForZeroShotObjectDetection,
                          SamModel, SamProcessor)
device = torch.device('cuda')

_gd_proc  = AutoProcessor.from_pretrained('IDEA-Research/grounding-dino-tiny')
_gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(
    'IDEA-Research/grounding-dino-tiny').to(device).eval()
_sam_proc  = SamProcessor.from_pretrained('facebook/sam-vit-base')
_sam_model = SamModel.from_pretrained('facebook/sam-vit-base').to(device).eval()

@torch.no_grad()
def predict_langsam(pil):
    # 1) GroundingDINO: detect "sea ice" boxes (text must be lowercase, end with '.')
    inp = _gd_proc(images=pil, text='sea ice.', return_tensors='pt').to(device)
    out = _gd_model(**inp)
    res = _gd_proc.post_process_grounded_object_detection(
        out, inp.input_ids, box_threshold=0.25, text_threshold=0.25,
        target_sizes=[pil.size[::-1]])
    boxes = res[0]['boxes']
    if boxes.numel() == 0:
        return np.zeros((pil.height, pil.width), bool)   # no detection (honest zero-shot result)
    # 2) SAM: segment from the detected boxes
    sinp = _sam_proc(pil, input_boxes=[boxes.cpu().tolist()], return_tensors='pt').to(device)
    sout = _sam_model(**sinp)
    masks = _sam_proc.image_processor.post_process_masks(
        sout.pred_masks.cpu(), sinp['original_sizes'].cpu(),
        sinp['reshaped_input_sizes'].cpu())[0]
    masks = np.asarray(masks).astype(bool)               # (n_boxes, n_per_box, H, W)
    if masks.ndim == 4:
        masks = masks[:, 0]                              # first mask per box
    return np.any(masks, axis=0)                         # union of all instances

res = evaluate_predictor(predict_langsam)
save_result('LangSAM', res)


## 4. DeepLabv3+ (trained on SAR, no text)

Standard semantic-segmentation baseline with **no** language input. Trained
briefly on your SAR train split, evaluated with the same harness. Shows what a
strong vision-only model achieves without text guidance.


In [ ]:
import torch, torch.nn as nn, numpy as np
import torch.nn.functional as F
from torchvision.models.segmentation import deeplabv3_resnet50
from torch.utils.data import Dataset, DataLoader
from PIL import Image

device = torch.device('cuda')

class _SegDS(Dataset):
    def __init__(self, split):
        from config import ICE_CLASSES, ICE_CLASS_TO_IDX
        from pathlib import Path
        self.items = []
        dr = Path(DATA_ROOT)
        for cls in ICE_CLASSES:
            idir, mdir = dr/cls/'images', dr/cls/'masks'
            if not idir.exists(): continue
            files = sorted(idir.glob('*.jpg'))
            rng = np.random.default_rng(42 + ICE_CLASS_TO_IDX[cls])
            order = rng.permutation(len(files)).tolist()
            n=len(files); ntr=round(n*0.7); nv=round(n*0.15)
            chosen = order[:ntr] if split=='train' else order[ntr+nv:]
            for i in chosen: self.items.append((files[i], mdir/files[i].name))
    def __len__(self): return len(self.items)
    def __getitem__(self, k):
        ip, mp = self.items[k]
        a = np.asarray(Image.open(ip).convert('RGB').resize((512,512))).astype('float32')/255.0
        a = (a-a.min())/(a.max()-a.min()+1e-8)
        x = torch.from_numpy(a.transpose(2,0,1))
        m = (np.array(Image.open(mp).convert('L').resize((512,512), Image.NEAREST))>127).astype('float32')             if mp.exists() else np.zeros((512,512),'float32')
        return x, torch.from_numpy(m).unsqueeze(0)

dl = DataLoader(_SegDS('train'), batch_size=4, shuffle=True, num_workers=2)
net = deeplabv3_resnet50(num_classes=1).to(device)
opt = torch.optim.AdamW(net.parameters(), lr=1e-4, weight_decay=1e-4)

print('Training DeepLabv3+ (15 epochs, no text)...')
net.train()
for ep in range(15):
    tot=0.0
    for x, y in dl:
        x, y = x.to(device), y.to(device)
        logit = net(x)['out']
        loss = F.binary_cross_entropy_with_logits(logit, y)              + (1 - (2*(torch.sigmoid(logit)*y).sum()+1)/((torch.sigmoid(logit)+y).sum()+1))
        opt.zero_grad(); loss.backward(); opt.step(); tot+=loss.item()
    print(f'  epoch {ep+1}/15  loss={tot/len(dl):.4f}')

net.eval()
@torch.no_grad()
def predict_deeplab(pil):
    a = np.asarray(pil.resize((512,512))).astype('float32')/255.0
    a = (a-a.min())/(a.max()-a.min()+1e-8)
    x = torch.from_numpy(a.transpose(2,0,1)).unsqueeze(0).to(device)
    return torch.sigmoid(net(x)['out']).squeeze().cpu().numpy() > 0.5

res = evaluate_predictor(predict_deeplab)
save_result('DeepLabV3plus', res)


## 5. LISA-7B (zero-shot reasoning segmentation)

⚠️ **Run in a FRESH session** (`Runtime → Restart session`, then re-run Section 0).
LISA pins an **older `transformers`** that conflicts with CLIPSeg/your model.

Query: *"Where is the sea ice in this synthetic aperture radar image? Please
output the segmentation mask."*

> LISA's API can vary by commit. If a call signature differs, the error will be
> explicit — fix the one line and re-run. We never fabricate the mask.


In [ ]:
import subprocess, sys, os
# Official LISA repo
if not os.path.exists('/content/LISA'):
    subprocess.run(['git','clone','https://github.com/dvlab-research/LISA','/content/LISA'], check=True)
subprocess.check_call([sys.executable,'-m','pip','install','-q',
    'transformers==4.31.0','sentencepiece','peft==0.4.0','bitsandbytes',
    'einops','timm','opencv-python'])
sys.path.insert(0, '/content/LISA')

import torch, numpy as np, cv2
from transformers import AutoTokenizer, CLIPImageProcessor
from model.LISA import LISAForCausalLM
from model.llava.mm_utils import tokenizer_image_token
from model.llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN
from model.segment_anything.utils.transforms import ResizeLongestSide

device = torch.device('cuda')
MODEL_ID = 'xinlai/LISA-7B-v1'
tok = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False)
tok.pad_token = tok.unk_token
seg_token_idx = tok('[SEG]', add_special_tokens=False).input_ids[0]

lisa = LISAForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,
    seg_token_idx=seg_token_idx, vision_tower='openai/clip-vit-large-patch14',
).to(device).eval()
lisa.get_model().initialize_vision_modules(lisa.get_model().config)
clip_proc = CLIPImageProcessor.from_pretrained('openai/clip-vit-large-patch14')
sam_tf = ResizeLongestSide(1024)

PROMPT = ('Where is the sea ice in this synthetic aperture radar image? '
          'Please output the segmentation mask.')

def _prep_sam(arr):
    x = sam_tf.apply_image(arr)
    x = torch.from_numpy(x).permute(2,0,1).contiguous()
    px_mean = torch.tensor([123.675,116.28,103.53]).view(-1,1,1)
    px_std  = torch.tensor([58.395,57.12,57.375]).view(-1,1,1)
    x = (x - px_mean)/px_std
    h,w = x.shape[-2:]
    x = torch.nn.functional.pad(x, (0,1024-w,0,1024-h))
    return x.unsqueeze(0).to(device).bfloat16()

@torch.no_grad()
def predict_lisa(pil):
    arr = np.asarray(pil.convert('RGB'))
    H,W = arr.shape[:2]
    conv = f"USER: {DEFAULT_IMAGE_TOKEN}\n{PROMPT} ASSISTANT:"
    ids = tokenizer_image_token(conv, tok, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).to(device)
    clip_img = clip_proc.preprocess(pil, return_tensors='pt')['pixel_values'][0].unsqueeze(0).to(device).bfloat16()
    sam_img = _prep_sam(arr)
    out = lisa.evaluate(clip_img, sam_img, ids,
                        resize_list=[sam_tf.get_preprocess_shape(H,W,1024)],
                        original_size_list=[(H,W)], max_new_tokens=128, tokenizer=tok)
    pred_masks = out[1] if isinstance(out, (tuple,list)) else out
    m = pred_masks[0]
    if hasattr(m,'detach'): m = m.detach().float().cpu().numpy()
    m = np.asarray(m)
    if m.ndim == 3: m = m[0]
    return m > 0

res = evaluate_predictor(predict_lisa)
save_result('LISA7B', res)


## 6. GeoPixel (zero-shot remote-sensing LMM)

⚠️ **Run in a FRESH session.** Built on InternLM-XComposer; may need `flash-attn`.
The closest-domain comparison (remote sensing). Query asks it to segment the
sea-ice region and emit the grounding mask.

> Follows the official GeoPixel inference flow. If their grounded-mask accessor
> differs by commit, the error will point to the exact line to adjust.


In [ ]:
import subprocess, sys, os
if not os.path.exists('/content/GeoPixel'):
    subprocess.run(['git','clone','https://github.com/mbzuai-oryx/GeoPixel','/content/GeoPixel'], check=True)
os.chdir('/content/GeoPixel')
subprocess.check_call([sys.executable,'-m','pip','install','-q','-r','requirements.txt'])
sys.path.insert(0, '/content/GeoPixel')

import torch, numpy as np
from transformers import AutoModel, AutoTokenizer
device = torch.device('cuda')

MODEL_ID = 'MBZUAI/GeoPixel-7B'   # check the repo README for the exact HF id
tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
gp  = AutoModel.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16,
                                trust_remote_code=True).to(device).eval()

QUERY = ('Please segment the sea ice region in this image and output the '
         'segmentation mask. <image>')

@torch.no_grad()
def predict_geopixel(pil):
    # GeoPixel exposes a grounded-generation call that returns text + pred_masks.
    # The exact method name is in their demo (e.g. model.generate / model.chat).
    out = gp.chat(tok, query=QUERY, image=pil, history=[],
                  max_new_tokens=256) if hasattr(gp, 'chat') else None
    masks = None
    if isinstance(out, (tuple, list)):
        for o in out:
            if hasattr(o, 'shape') or isinstance(o, (list, np.ndarray)):
                masks = o
    elif isinstance(out, dict):
        masks = out.get('pred_masks') or out.get('masks')
    if masks is None:
        return np.zeros((pil.height, pil.width), bool)
    m = np.asarray(masks[0] if len(np.shape(masks)) > 2 else masks)
    if hasattr(m, 'detach'): m = m.detach().float().cpu().numpy()
    if m.ndim == 3: m = m[0]
    return m > 0.5

res = evaluate_predictor(predict_geopixel)
save_result('GeoPixel', res)


## 7. SEEM (zero-shot, text-promptable)

⚠️ **Run in a FRESH session.** Heavy install (detectron2-style deps).
Segment-Everything-Everywhere with the text prompt "sea ice".


In [ ]:
import subprocess, sys, os
if not os.path.exists('/content/SEEM'):
    subprocess.run(['git','clone',
        'https://github.com/UX-Decoder/Segment-Everything-Everywhere-All-At-Once',
        '/content/SEEM'], check=True)
os.chdir('/content/SEEM/demo_code' if os.path.exists('/content/SEEM/demo_code') else '/content/SEEM')
subprocess.check_call([sys.executable,'-m','pip','install','-q','-r',
    '/content/SEEM/assets/requirements/requirements.txt'])
sys.path.insert(0, '/content/SEEM')

import torch, numpy as np
# SEEM loads a config + checkpoint; the demo exposes a text-prompted interface.
# Follow the official demo (xdecoder/SEEM) build_model + inference flow.
from modeling.BaseModel import BaseModel
from modeling import build_model
from utils.arguments import load_opt_from_config_files
from utils.distributed import init_distributed

device = torch.device('cuda')
opt = load_opt_from_config_files(['/content/SEEM/configs/seem/focall_unicl_lang_v1.yaml'])
opt = init_distributed(opt)
CKPT = '/content/seem_focall_v1.pt'
if not os.path.exists(CKPT):
    subprocess.run(['wget','-q','-O',CKPT,
        'https://huggingface.co/xdecoder/SEEM/resolve/main/seem_focall_v1.pt'], check=True)
seem = BaseModel(opt, build_model(opt)).from_pretrained(CKPT).to(device).eval()

@torch.no_grad()
def predict_seem(pil):
    # Text-prompted grounding; SEEM demo provides inference_seem_pano/interactive.
    # We request the "sea ice" grounding mask. Adapt accessor to the demo helper.
    from demo.seem.tasks import interactive_infer_image  # demo helper
    res = interactive_infer_image(seem, None, pil, tasks=['Text'],
                                  reftxt='sea ice')
    m = res[1] if isinstance(res, (tuple,list)) else res
    m = np.asarray(m)
    if m.ndim == 3: m = m[0]
    return m > 0.5

res = evaluate_predictor(predict_seem)
save_result('SEEM', res)


## 8. 📊 Aggregate → Comparison Table + LaTeX

Reads every `*.json` saved in the Drive comparison folder and builds the table.
Run this in any session after you've collected the model results you want.


In [ ]:
import json, glob
import pandas as pd

rows = []
for f in sorted(glob.glob(f'{COMPARE_DIR}/*.json')):
    d = json.load(open(f))
    rows.append(d)

if not rows:
    print('No results yet. Run at least one model section.')
else:
    PRETTY = {'OursTextGuided':'Ours (text-guided)', 'CLIPSeg':'CLIPSeg',
              'LangSAM':'lang-sam (GD+SAM)', 'DeepLabV3plus':'DeepLabv3+ (no text)',
              'LISA7B':'LISA-7B', 'GeoPixel':'GeoPixel', 'SEEM':'SEEM'}
    df = pd.DataFrame(rows)
    df['Model'] = df['model'].map(lambda m: PRETTY.get(m, m))
    df = df[['Model','gIoU','cIoU','Dice','n']].copy()
    for c in ['gIoU','cIoU','Dice']:
        df[c] = (df[c]*100).round(2)
    # Order: baselines first, ours last (highlighted)
    df['_ours'] = df['Model'].str.startswith('Ours')
    df = df.sort_values(['_ours','gIoU']).drop(columns='_ours').reset_index(drop=True)
    print(df.to_string(index=False))

    # ── LaTeX (IEEE-style, caption above) ─────────────────────────────────────
    print('\n' + '='*70 + '\nLaTeX:\n' + '='*70)
    lines = [r'\begin{table}[t]', r'\centering',
             r'\caption{Comparison on the SAR sea ice test set ('
             + f'{int(df["n"].iloc[0])}' + r' images). All models evaluated with '
             r'identical metric code. gIoU/cIoU/Dice in \%.}',
             r'\label{tab:comparison}',
             r'\begin{tabular}{lccc}', r'\hline',
             r'Method & gIoU & cIoU & Dice \\', r'\hline']
    for _, r in df.iterrows():
        name = r['Model']
        if name.startswith('Ours'):
            lines.append(rf"\textbf{{{name}}} & \textbf{{{r.gIoU}}} & "
                         rf"\textbf{{{r.cIoU}}} & \textbf{{{r.Dice}}} \\")
        else:
            lines.append(rf"{name} & {r.gIoU} & {r.cIoU} & {r.Dice} \\")
    lines += [r'\hline', r'\end{tabular}', r'\end{table}']
    print('\n'.join(lines))

    df.to_csv(f'{COMPARE_DIR}/comparison_table.csv', index=False)
    print('\nSaved CSV →', f'{COMPARE_DIR}/comparison_table.csv')
